In [1]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
import glob
warnings.filterwarnings("ignore")

### Nielsen

In [2]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
xls = pd.ExcelFile(f'{dir}/../Data Source/Nielsen/Nielsen Brand Ranking_Jun26 v270726.xlsx')
sheet_names = ['MY Female CPD', 'MY Male CPD', 'MY Cosmetics']
dfs = {}
# Read each sheet into a DataFrame and store it in the dictionary
for sheet_name in sheet_names:
    dfs[sheet_name] = pd.read_excel(xls, sheet_name=sheet_name)

In [3]:
for df in dfs:
    print(df)

MY Female CPD
MY Male CPD
MY Cosmetics


In [4]:
# Brand Mapping
def map_brand(row):
    if pd.notna(row['BRAND']):
        return row['BRAND']
    else:
        return row['MANUFACTURER']

In [5]:
# Get Current Year, Last Year and Year before last year
current_year = 2026
last_year = current_year - 1
year_before_last_year = current_year - 2
print(current_year, last_year, year_before_last_year)

2026 2025 2024


In [6]:
# # Year Transformation
def map_year(row):
    if 'Cal Yr' in row['Periods']:
        year = pd.Series(row['Periods']).str.extract(r'(\d{4})').values[0]
        return year.astype(str)[0]
    
    elif 'YTD YA' in row['Periods']:
            return 'YTD ' + str(last_year)
    
    elif 'YTD' in row['Periods']:
            return 'YTD ' + str(current_year)

# import re

# def map_year(row):
#     period = row['Periods']

#     # YTD last year
#     if 'YTD YA' in period:
#         return f'YTD {last_year}'

#     # YTD current year
#     elif 'YTD' in period:
#         return f'YTD {current_year}'

#     # Calendar year
#     elif 'Cal Yr' in period:
#         year = re.search(r'\d{4}', period).group()
#         return year

In [7]:
# Axes Category Mapping
def map_axes(row):
    if ('LOREAL_CATEGORY' in row.index) and (row['LOREAL_CATEGORY'] == 'COSMETIC'):
        return 'Cosmetics'
    elif ('GENDER' in row.index) and (row['GENDER'] == 'WOMAN'):
        return 'Female Skincare'
    elif ('GENDER' in row.index) and (row['GENDER'] == 'MEN'):
        return 'Male Skincare'

In [8]:
# # Important!! Try to understand the filter and logic here
for df in dfs:
    
    dfs[df]['Year'] = dfs[df].apply(map_year, axis=1)
    dfs[df]['Mass/Mass medic'] = 'Mass'
    dfs[df]['Brand'] = dfs[df].apply(map_brand, axis=1)
    dfs[df]['Axes'] = dfs[df].apply(map_axes, axis=1)


In [9]:
print(dfs[df].columns)
print(dfs[df].head())

Index(['Markets', 'Periods', 'LOREAL_CATEGORY', 'MANUFACTURER', 'BRAND',
       'Sales Value', 'Year', 'Mass/Mass medic', 'Brand', 'Axes'],
      dtype='str')
                      Markets                Periods LOREAL_CATEGORY  \
0  Total Malaysia Key Account  YTD - 26 w/e 28/06/26        COSMETIC   
1  Total Malaysia Key Account  YTD - 26 w/e 28/06/26        COSMETIC   
2  Total Malaysia Key Account  YTD - 26 w/e 28/06/26        COSMETIC   
3  Total Malaysia Key Account  YTD - 26 w/e 28/06/26        COSMETIC   
4  Total Malaysia Key Account  YTD - 26 w/e 28/06/26        COSMETIC   

      MANUFACTURER       BRAND   Sales Value      Year Mass/Mass medic  \
0  EXCLUSIVE BRAND         NaN  6.003520e+07  YTD 2026            Mass   
1    PRIVATE LABEL         NaN  3.709896e+06  YTD 2026            Mass   
2              NaN        1028  7.169400e+03  YTD 2026            Mass   
3              NaN     16BRAND  2.970000e+02  YTD 2026            Mass   
4              NaN  3 BEAUTIES  6.3479

In [10]:
for df in dfs:
    # Pivot
    dfs[df] = dfs[df].pivot_table(
        index=['Mass/Mass medic', 'Axes', 'Brand'],
        columns='Year',
        values='Sales Value',
        aggfunc='sum'
    )

    dfs[df] = dfs[df].reset_index()

    # Get available columns
    cols = dfs[df].columns.tolist()

    # Define desired columns
    desired_cols = [
        'Mass/Mass medic',
        'Axes',
        'Brand',
        str(year_before_last_year),   # e.g. 2023
        str(last_year),               # e.g. 2025
        f'YTD {last_year}',           # YTD 2025
        f'YTD {current_year}'         # YTD 2026
    ]

    # Keep only columns that actually exist (prevents KeyError)
    final_cols = [col for col in desired_cols if col in cols]

    dfs[df] = dfs[df][final_cols]

    # Fill missing values
    dfs[df] = dfs[df].fillna(0)

    # Optional: sort columns nicely (YTD at the end)
    non_ytd = [col for col in final_cols if 'YTD' not in col and col not in ['Mass/Mass medic', 'Axes', 'Brand']]
    ytd = [col for col in final_cols if 'YTD' in col]

    dfs[df] = dfs[df][['Mass/Mass medic', 'Axes', 'Brand'] + sorted(non_ytd) + sorted(ytd)]

    # Preview
    print(f"\nProcessed: {df}")
    display(dfs[df].head(3))


Processed: MY Female CPD


Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Female Skincare,& HONEY,214388.286,185033.508,109779.802,36894.170
1,Mass,Female Skincare,1028,304.300,0.000,0.000,0.000
2,Mass,Female Skincare,3650,0.000,748.920,376.920,749.111



Processed: MY Male CPD


Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Male Skincare,ADIDAS,30.000,60.000,15.000,30.000
1,Mass,Male Skincare,BAD LAB,5176958.681,4039855.347,2144585.490,1784339.300
2,Mass,Male Skincare,CODE 10,187916.962,152964.683,68335.769,103777.387



Processed: MY Cosmetics


Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Cosmetics,1028,54308.61,29663.15,17302.70,7169.4
1,Mass,Cosmetics,16BRAND,0.00,173.25,0.00,297.0
2,Mass,Cosmetics,3 BEAUTIES,76526.80,122173.65,43604.15,63479.2


In [11]:
nielsen = pd.concat(dfs.values())
# Data Checking
nielsen.head()

Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Female Skincare,& HONEY,214388.286,185033.508,109779.802,36894.170
1,Mass,Female Skincare,1028,304.300,0.000,0.000,0.000
2,Mass,Female Skincare,3650,0.000,748.920,376.920,749.111
3,Mass,Female Skincare,3W CLINIC,56608.527,17120.986,11176.347,3481.211
4,Mass,Female Skincare,701,5879.173,14612.962,5556.473,7576.392


### OMT

In [12]:
base_dir = 'C:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - JUNE\\loreal-report-automation (2)\\Brand Ranking'


In [13]:
# Important!! Make sure the files exist and are updated first
omt_dir = os.path.join(base_dir, '..\\Data Source\\OMT - O+O\\MY CPD')
omt_start_month = pd.Period('2024-01', freq='M')
omt_pattern = os.path.join(omt_dir, 'OMT MY CPD *.xlsx')
available_omt_months = []

for file in glob.glob(omt_pattern):
    month_text = os.path.splitext(os.path.basename(file))[0].replace('OMT MY CPD ', '')
    try:
        available_omt_months.append(pd.Period(month_text, freq='M'))
    except ValueError:
        pass

if not available_omt_months:
    raise FileNotFoundError(f'No OMT files found in {omt_dir}')

omt_end_month = max(available_omt_months)
omt_months = pd.period_range(omt_start_month, omt_end_month, freq='M')
print(f'Latest OMT month found: {omt_end_month}')
omt_files = [os.path.join(omt_dir, f'OMT MY CPD {month}.xlsx') for month in omt_months]

missing_files = [file for file in omt_files if not os.path.exists(file)]
if missing_files:
    raise FileNotFoundError('Missing OMT files:\n' + '\n'.join(missing_files))

print(f'Reading {len(omt_files)} OMT files')

omt_raw = pd.concat([pd.read_excel(file, sheet_name='Export', keep_default_na=False) for file in omt_files], axis=0)

# Rename the literal "na" brand
omt_raw['Brand'] = omt_raw['Brand'].apply(
    lambda x: 'NA Brand'
    if isinstance(x, str) and x.strip().lower() == 'na'
    else x
)

omt_wg = omt_raw.copy()
omt_f = omt_raw.copy()
print('Done reading OMT files')

mapping = pd.read_excel(f'{dir}/../Data Source/CPD Skincare Mapping/Skincare Mapping.xlsx', sheet_name='MY')

Latest OMT month found: 2026-06
Reading 30 OMT files
Done reading OMT files


In [14]:
# Structure the df into dictionary
omt = {
    'omt_wg': omt_wg,
    'omt_f': omt_f
}

In [15]:
# Brand Mapping
def map_brand_group(row):
    if row['Brand'] in ["GARNIER", "MAYBELLINE","3CE"]:
        return row['Brand']
    elif row['Brand'] == "L'OREAL PARIS":
        return 'LOREAL PARIS'
    else:
        return 'Market'

In [16]:
# Important!! Try to understand the filter and logic here
for df in omt:
    omt[df] = omt[df][omt[df]['Category L1'] != 'FRAGRANCE']
    omt[df] = omt[df][omt[df]['Category L2'].isin(['EYE MAKEUP','FACE MAKEUP','LIP MAKEUP', 'NAIL MAKEUP', 'OTHER MAKEUP','FACE CARE & CLEANSING','SUN CARE','HAIR COLOR','HAIR CARE'])]
    omt[df] = omt[df][(omt[df]['Category L2'] != 'SUN CARE') | (omt[df]['Category L3'] == 'FACE PROTECTION')]
    omt[df][['Year', 'Month']] = omt[df]['Year Month'].str.split('-', expand=True)
    omt[df] = omt[df].dropna(subset=['Year'])
    omt[df]['Subdivision'] = 'NA'
    omt[df][['Year', 'Month']] = omt[df][['Year', 'Month']].astype(int)
    omt[df]['brand_group'] = omt[df].apply(map_brand_group, axis=1)
    omt[df] = omt[df][['Mall Type', 'brand_group', 'Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2','Total Est. Sales Local']]
    omt[df].reset_index(drop=True, inplace=True)
    #omt[df]['Subcategory'] = ''
    omt[df] = omt[df].groupby(['Mall Type', 'brand_group', 'Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2'])[['Total Est. Sales Local']].sum().reset_index()
    omt[df].reset_index(drop=True, inplace=True)
    omt[df] = omt[df].sort_values(by=['Year', 'Month'])

#### OMT Skincare

In [17]:
# Skincare Percentage Share for Female and Male
omt_wg_sc = omt['omt_wg'][omt['omt_wg']['Category L1'] == 'SKIN CARE']
omt_f_sc = omt['omt_f'][omt['omt_f']['Category L1'] == 'SKIN CARE']
omt_sc = {
    'omt_wg_sc': omt_wg_sc,
    'omt_f_sc': omt_f_sc
}

In [18]:
# Seperate for wg and f
omt_wg_sc = omt_sc['omt_wg_sc']
omt_f_sc = omt_sc['omt_f_sc']
omt_wg_sc

,Mall Type,brand_group,Brand,Year,Month,Universe,Subdivision,Category L1,Category L2,Total Est. Sales Local
126,Lazada Mall,GARNIER,GARNIER,2024,1,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,57873.43
127,Lazada Mall,GARNIER,GARNIER,2024,1,MASS,NA,SKIN CARE,SUN CARE,1372.92
248,Lazada Mall,LOREAL PARIS,L'OREAL PARIS,2024,1,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,143335.27
249,Lazada Mall,LOREAL PARIS,L'OREAL PARIS,2024,1,MASS,NA,SKIN CARE,SUN CARE,10590.94
468,Lazada Mall,MAYBELLINE,MAYBELLINE,2024,1,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,4310.88
...,...,...,...,...,...,...,...,...,...,...
183225,Tiktok Mall,Market,ZARZOU,2026,6,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,18170.26
183226,Tiktok Mall,Market,ZARZOU,2026,6,MASS,NA,SKIN CARE,SUN CARE,2349.6
183320,Tiktok Mall,Market,ZIGTAG,2026,6,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,1992.1
183368,Tiktok Mall,Market,ZOZU,2026,6,MASS,NA,SKIN CARE,FACE CARE & CLEANSING,12.32


In [19]:
omt_f_sc.info()

<class 'pandas.DataFrame'>
Index: 84873 entries, 126 to 183385
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Mall Type               84873 non-null  str   
 1   brand_group             84873 non-null  str   
 2   Brand                   84873 non-null  str   
 3   Year                    84873 non-null  int64 
 4   Month                   84873 non-null  int64 
 5   Universe                84873 non-null  str   
 6   Subdivision             84873 non-null  str   
 7   Category L1             84873 non-null  str   
 8   Category L2             84873 non-null  str   
 9   Total Est. Sales Local  84873 non-null  object
dtypes: int64(2), object(1), str(7)
memory usage: 7.1+ MB


In [20]:
omt_wg_sc.info()

<class 'pandas.DataFrame'>
Index: 84873 entries, 126 to 183385
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Mall Type               84873 non-null  str   
 1   brand_group             84873 non-null  str   
 2   Brand                   84873 non-null  str   
 3   Year                    84873 non-null  int64 
 4   Month                   84873 non-null  int64 
 5   Universe                84873 non-null  str   
 6   Subdivision             84873 non-null  str   
 7   Category L1             84873 non-null  str   
 8   Category L2             84873 non-null  str   
 9   Total Est. Sales Local  84873 non-null  object
dtypes: int64(2), object(1), str(7)
memory usage: 7.1+ MB


In [21]:
# Join with Mapping sheet using SQL query
query_1 = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                Tiktok,
                'FEMALE SKINCARE' AS Category
            FROM mapping
        
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                1 - Tiktok AS Tiktok,
                'MALE SKINCARE' AS Category
            FROM mapping
        )

        SELECT
            a.Brand,
            a.Year,
            a.Month,
            a.Universe,
            a.Subdivision,
            b.Category AS "Category L1",
            a."Category L2",
            CASE
                WHEN a."Mall Type" = 'Shopee Mall' THEN a."Total Est. Sales Local" * b.Shopee
                WHEN a."Mall Type" = 'Lazada Mall' THEN a."Total Est. Sales Local" * b.Lazada
                WHEN a."Mall Type" = 'Tiktok Mall' THEN a."Total Est. Sales Local" * b.Tiktok
            END AS "Total Est. Sales Local"
        FROM omt_wg_sc a
            LEFT JOIN mapping_tx b
                ON (
                    a.brand_group = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

In [22]:
# Join with Mapping sheet using SQL query
query_2 = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                Tiktok,
                'FEMALE SKINCARE' AS Category
            FROM mapping
        
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                1 - Tiktok AS Tiktok,
                'MALE SKINCARE' AS Category
            FROM mapping
        )

        SELECT
            a.Brand,
            a.Year,
            a.Month,
            a.Universe,
            a.Subdivision,
            b.Category AS "Category L1",
            a."Category L2",
            CASE
                WHEN a."Mall Type" = 'Shopee Mall' THEN a."Total Est. Sales Local" * b.Shopee
                WHEN a."Mall Type" = 'Lazada Mall' THEN a."Total Est. Sales Local" * b.Lazada
                WHEN a."Mall Type" = 'Tiktok Mall' THEN a."Total Est. Sales Local" * b.Tiktok
            END AS "Total Est. Sales Local"
        FROM omt_f_sc a
            LEFT JOIN mapping_tx b
                ON (
                    a.brand_group = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

In [23]:
# run the query and store the result in a new dataframe
omt_wg_sc = sqldf(query_1)
omt_f_sc = sqldf(query_2)

#### OMT Hair and Makeup

In [24]:
# Select the columns for Hair and Makeup
for df in omt:
    omt[df] = omt[df][['Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2','Total Est. Sales Local']]
    omt[df].reset_index(drop=True, inplace=True)

In [25]:
# Exclude Skincare
omt_wg_hm = omt['omt_wg'][omt['omt_wg']['Category L1'] != 'SKIN CARE']
omt_f_hm = omt['omt_f'][omt['omt_f']['Category L1'] != 'SKIN CARE']

#### Merge Skincare and Hair and Makeup

In [26]:
omt_wg = pd.concat([omt_wg_sc, omt_wg_hm], ignore_index=True)
omt_f = pd.concat([omt_f_sc, omt_f_hm], ignore_index=True)

### Seperating Data into Different Tabs

In [27]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()
year = last_month.year

# Format the output as "MMM YYYY"
# filemonth = last_month.strftime("%b %Y").upper()
filemonth = "JUNE 2026"
# Print the result
print(f"{filemonth}")

JUNE 2026


In [28]:
if not os.path.exists(f'../Generated Data/Brand Ranking/{filemonth}'):
        os.makedirs(f'../Generated Data/Brand Ranking/{filemonth}')

In [29]:
with pd.ExcelWriter(f'../Generated Data/Brand Ranking/{filemonth}/MY CPD Brand Ranking {filemonth}.xlsx', engine='xlsxwriter') as writer:
    nielsen.to_excel(writer, sheet_name='Nielsen', index=False)
    omt_wg.to_excel(writer, sheet_name='OMT-WG', index=False)
    omt_f.to_excel(writer, sheet_name='OMT-F', index=False)

In [30]:
# import pandas as pd

# def clean_brand_data(input_file, output_file):
#     # Read the file into a DataFrame
#     df = pd.read_csv(input_file)
    
#     # Ensure the 'Brand' column exists
#     if 'Brand' in df.columns:
#         # Remove rows where 'Brand' column contains '-'
#         df_cleaned = df[df['Brand'] != '-']
        
#         # Save the cleaned data to a new file
#         df_cleaned.to_csv(output_file, index=False)
#         print(f"Cleaned data saved to {output_file}")
#     else:
#         print("Error: 'Brand' column not found in the file.")

# # Example usage
# input_file = "CPD Brand Ranking Data.csv"  # Replace with your file name
# output_file = "cleaned_data.csv"
# clean_brand_data(input_file, output_file)